# **InsightViewer — Demo Pipeline**

End-to-end walkthrough of the four pipeline stages:

| # | Stage | Input | Output |
|---|-------|-------|--------|
| 1 | **Ingest** | PDF / Excel / CSV | Chroma (chunks) + DuckDB (metrics) |
| 2 | **Retrieve** | Natural-language query | Ranked text chunks |
| 3 | **Query** | Ticker / metric name | Structured metrics DataFrame |
| 4 | **Visualize** | DataFrames | Matplotlib charts |

A final cell wires all four stages into a single `run_pipeline()` call.

## **Setup**

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import re
import uuid
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, List, Optional

import duckdb
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd
import pdfplumber
import pymupdf4llm
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

try:
    from langchain_chroma import Chroma
except ImportError:
    from langchain_community.vectorstores import Chroma  # type: ignore

try:
    from langchain_community.embeddings import SentenceTransformerEmbeddings
except ImportError:
    from langchain.embeddings import SentenceTransformerEmbeddings  # type: ignore

ROOT = Path('/home/ssever/InsightViewer')

@dataclass
class DemoConfig:
    data_dir: Path = ROOT / 'data' / 'test'
    stock_csv: Path = ROOT / 'data' / 'sql' / 'MSFT_1986-03-13_2025-02-04.csv'
    chroma_dir: Path = ROOT / 'storage' / 'chroma'
    duckdb_path: Path = ROOT / 'storage' / 'metrics.duckdb'
    collection_name: str = 'filings'
    embed_model: str = os.getenv('EMBED_MODEL', 'all-MiniLM-L12-v2')
    chunk_size: int = int(os.getenv('CHUNK_SIZE', '900'))
    chunk_overlap: int = int(os.getenv('CHUNK_OVERLAP', '120'))
    default_ticker: str = 'MSFT'
    default_request: str = 'What drove revenue growth, and can you show the relevant trend?'
    top_k: int = 5

CFG = DemoConfig()
#print(asdict(CFG))
print("Setup complete.")

## **Ingest**

Three data sources are handled here:

- **PDF** → markdown → chunks → Chroma; tables → metrics → DuckDB
- **Excel / CSV** → pandas DataFrame (used directly in visualizations)

Helper functions are defined in sub-cells, then applied to the files in `data/test/`.

In [ ]:
# PDF -> Markdown -> Chunks -> Chroma

TARGET_METRICS: Dict[str, List[str]] = {
    'revenue': ['revenue', 'net revenue', 'total revenue', 'net sales'],
    'net_income': ['net income', 'net earnings', 'profit for the year', 'net profit'],
}

def filename_meta(path: Path) -> dict:
    stem = path.stem
    m_tick = re.match(r'^([A-Z]{1,6})', stem)
    m_fy = re.search(r'FY(\d{2,4})', stem, re.I)
    m_q = re.search(r'Q(\d)', stem, re.I)
    m_form = re.search(r'(10Q|10K|8K)', stem, re.I)
    fy_raw = m_fy.group(1) if m_fy else None
    return {
        'ticker': m_tick.group(1) if m_tick else None,
        'fiscal_year': ('20' + fy_raw if fy_raw and len(fy_raw) == 2 else fy_raw),
        'fiscal_period': f"Q{m_q.group(1)}" if m_q else None,
        'filing_type': {'10Q': '10-Q', '10K': '10-K', '8K': '8-K'}.get(m_form.group(1).upper()) if m_form else None,
        'source_path': str(path),
        'source_filename': path.name,
    }

def pdf_to_markdown(path: Path) -> str:
    return pymupdf4llm.to_markdown(str(path))

def md_to_chunks(md: str, meta: dict, *, chunk_size: int, chunk_overlap: int) -> List[dict]:
    header_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=[('#', 'h1'), ('##', 'h2'), ('###', 'h3')]
    )
    chunks: List[dict] = []
    for doc in header_splitter.split_text(md):
        text = doc.page_content
        m = {**meta, **doc.metadata}
        m['has_table'] = bool(re.search(r'(^|\n)\s*\|.+\|\s*(\n|$)', text))
        if m['has_table'] or len(text) <= chunk_size:
            chunks.append({'text': text, 'metadata': m})
        else:
            splitter = RecursiveCharacterTextSplitter(
                chunk_size=chunk_size,
                chunk_overlap=chunk_overlap,
            )
            for t in splitter.split_text(text):
                chunks.append({'text': t, 'metadata': m})
    return chunks

def get_vectorstore(cfg: DemoConfig) -> Chroma:
    emb = SentenceTransformerEmbeddings(model_name=cfg.embed_model)
    return Chroma(
        collection_name=cfg.collection_name,
        persist_directory=str(cfg.chroma_dir),
        embedding_function=emb,
    )

def upsert_chunks(cfg: DemoConfig, chunks: List[dict]) -> int:
    if not chunks:
        return 0
    vs = get_vectorstore(cfg)
    texts, metas, ids = [], [], []
    for idx, chunk in enumerate(chunks):
        meta = {
            k: str(v) if not isinstance(v, (str, int, float, bool)) else v
            for k, v in chunk['metadata'].items() if v is not None
        }
        meta['chunk_index'] = idx
        texts.append(chunk['text'])
        metas.append(meta)
        ids.append(hashlib.sha1(f"{meta.get('source_path', '')}::{idx}".encode()).hexdigest())
    vs.add_texts(texts=texts, metadatas=metas, ids=ids)
    try:
        vs.persist()  # type: ignore[attr-defined]
    except Exception:
        pass
    return len(texts)


print("PDF -> Chroma helpers ready.")

In [ ]:
# PDF tables -> tidy metrics -> DuckDB

def _parse_number(cell: Any) -> Optional[float]:
    s = str(cell).strip().replace(',', '')
    neg = s.startswith('(') and s.endswith(')')
    if neg:
        s = s[1:-1]
    s = re.sub(r'[^\d.\-]', '', s)
    try:
        value = float(s)
        return -value if neg else value
    except (ValueError, TypeError):
        return None

def _best_metric(label: str) -> Optional[str]:
    from rapidfuzz import fuzz, process
    normalized = re.sub(r'\s+', ' ', label.lower()).strip()
    best_norm, best_score = None, 0
    for norm, synonyms in TARGET_METRICS.items():
        result = process.extractOne(normalized, synonyms, scorer=fuzz.token_sort_ratio)
        if result and result[1] > best_score:
            best_norm, best_score = norm, result[1]
    return best_norm if best_score >= 80 else None

_DDL = '''
CREATE TABLE IF NOT EXISTS metrics (
    id TEXT PRIMARY KEY,
    ticker TEXT,
    filing_type TEXT,
    fiscal_year INTEGER,
    fiscal_period TEXT,
    metric TEXT,
    period_year INTEGER,
    value DOUBLE,
    units TEXT,
    scale INTEGER,
    source_filename TEXT,
    page INTEGER,
    table_id INTEGER,
    provenance JSON
)
'''

def init_db(cfg: DemoConfig) -> duckdb.DuckDBPyConnection:
    conn = duckdb.connect(str(cfg.duckdb_path))
    conn.execute(_DDL)
    return conn

def insert_metrics(conn: duckdb.DuckDBPyConnection, rows: List[dict]) -> int:
    if not rows:
        return 0
    frame = pd.DataFrame(rows)
    conn.register('_rows', frame)
    conn.execute('INSERT OR REPLACE INTO metrics SELECT * FROM _rows')
    conn.unregister('_rows')
    return len(rows)

def extract_metrics(path: Path, meta: dict) -> List[dict]:
    md_text = pdf_to_markdown(path)
    units = 'USD' if ('$' in md_text or 'usd' in md_text.lower()) else None
    scale = 1_000_000 if 'in millions' in md_text.lower() else 1_000 if 'in thousands' in md_text.lower() else None
    rows: List[dict] = []
    with pdfplumber.open(str(path)) as pdf:
        for page_idx, page in enumerate(pdf.pages):
            for table_idx, table in enumerate(page.extract_tables() or []):
                if not table or len(table) < 2:
                    continue
                df = pd.DataFrame(table)
                header = df.iloc[0].astype(str).tolist()
                if sum(bool(re.search(r'[A-Za-z]', c)) for c in header) >= 2:
                    df.columns = header
                    df = df.iloc[1:].reset_index(drop=True)
                label_col = df.columns[0]
                year_cols = {
                    c: re.search(r'(20\d{2})', str(c)).group(1)
                    for c in df.columns if re.search(r'(20\d{2})', str(c))
                }
                if not year_cols:
                    continue
                for _, row in df.iterrows():
                    label = str(row[label_col])
                    metric = _best_metric(label)
                    if not metric:
                        continue
                    for column, year in year_cols.items():
                        value = _parse_number(row[column])
                        if value is None:
                            continue
                        rows.append({
                            'id': str(uuid.uuid4()),
                            'ticker': meta.get('ticker'),
                            'filing_type': meta.get('filing_type'),
                            'fiscal_year': int(meta['fiscal_year']) if meta.get('fiscal_year') else None,
                            'fiscal_period': meta.get('fiscal_period'),
                            'metric': metric,
                            'period_year': int(year),
                            'value': float(value * (scale or 1)),
                            'units': units,
                            'scale': scale,
                            'source_filename': path.name,
                            'page': page_idx + 1,
                            'table_id': table_idx + 1,
                            'provenance': json.dumps({'label_raw': label}),
                        })
    return rows

def load_tabular_data(cfg: DemoConfig) -> dict:
    excel_candidates = sorted(cfg.data_dir.glob('*.xlsx'))
    dividend_path = excel_candidates[0] if excel_candidates else None
    df_div = pd.read_excel(dividend_path) if dividend_path and dividend_path.exists() else pd.DataFrame()
    if not df_div.empty:
        df_div.columns = [c.strip().lower().replace(' ', '_') for c in df_div.columns]
    df_stock = pd.read_csv(cfg.stock_csv, parse_dates=['Date']) if cfg.stock_csv.exists() else pd.DataFrame()
    return {
        'dividends': df_div,
        'stock_prices': df_stock,
        'dividend_path': str(dividend_path) if dividend_path else None,
        'stock_path': str(cfg.stock_csv) if cfg.stock_csv.exists() else None,
    }

print("Metrics + DuckDB helpers ready.")

In [ ]:
# Simple routing based on request content

def route_request(user_request: str) -> dict:
    text = user_request.lower()
    wants_explanation = any(term in text for term in ['why', 'what drove', 'explain', 'discuss'])
    wants_trend = any(term in text for term in ['trend', 'chart', 'visualize', 'show'])
    wants_metrics = any(term in text for term in ['revenue', 'income', 'metric', 'compare', 'growth'])

    route = {
        'semantic_search': wants_explanation or not wants_metrics,
        'metrics_query': wants_metrics,
        'spreadsheet_lookup': wants_trend,
        'visualize': wants_trend,
    }

    if not any(route.values()):
        route['semantic_search'] = True

    return route

sample_requests = [
    'What drove revenue growth in Q1 FY2025?',
    'Show a revenue and stock-price trend for Microsoft.',
    'Compare revenue and net income over time.',
]

for request in sample_requests:
    print(request)
    print(route_request(request))
    print('-' * 60)

In [ ]:
conn = init_db(CFG)
tabular = load_tabular_data(CFG)

pdf_files = sorted(CFG.data_dir.glob('*.pdf'))
ingest_report = []

for pdf_path in pdf_files:
    meta = filename_meta(pdf_path)
    markdown = pdf_to_markdown(pdf_path)
    chunks = md_to_chunks(markdown, meta, chunk_size=CFG.chunk_size, chunk_overlap=CFG.chunk_overlap)
    metric_rows = extract_metrics(pdf_path, meta)

    chunk_count = upsert_chunks(CFG, chunks)
    metric_count = insert_metrics(conn, metric_rows)

    ingest_report.append({
        'file': pdf_path.name,
        'ticker': meta.get('ticker'),
        'chunks_upserted': chunk_count,
        'metrics_inserted': metric_count,
    })

df_ingest = pd.DataFrame(ingest_report)
display(df_ingest if not df_ingest.empty else pd.DataFrame(columns=['file', 'ticker', 'chunks_upserted', 'metrics_inserted']))

print('Dividend rows:', len(tabular['dividends']))
print('Stock rows:', len(tabular['stock_prices']))

In [ ]:
def retrieve_text_evidence(cfg: DemoConfig, user_request: str, k: int) -> List[dict]:
    vs = get_vectorstore(cfg)
    results = vs.similarity_search_with_score(user_request, k=k)
    evidence = []
    for rank, (doc, score) in enumerate(results, start=1):
        evidence.append({
            'rank': rank,
            'score': float(score),
            'source_filename': doc.metadata.get('source_filename'),
            'headers': [doc.metadata.get('h1'), doc.metadata.get('h2'), doc.metadata.get('h3')],
            'preview': doc.page_content[:500],
            'document': doc,
        })
    return evidence

def fetch_metrics(conn: duckdb.DuckDBPyConnection, ticker: str) -> pd.DataFrame:
    return conn.execute(
        '''
        SELECT
            ticker,
            metric,
            period_year,
            ROUND(SUM(value) / 1e9, 2) AS value_billions
        FROM metrics
        WHERE ticker = ?
        GROUP BY ticker, metric, period_year
        ORDER BY metric, period_year
        ''',
        [ticker],
    ).df()

def build_chart_payload(metrics_df: pd.DataFrame, tables: dict) -> dict:
    payload = {
        'metrics': metrics_df.copy(),
        'dividends': tables['dividends'].copy(),
        'stock_prices': tables['stock_prices'].copy(),
    }
    return payload

def build_answer_stub(request: str, route: dict, evidence: List[dict], metrics_df: pd.DataFrame) -> str:
    parts = [
        f'Request: {request}',
        f'Route: {route}',
        f'Text evidence count: {len(evidence)}',
        f'Metric rows: {len(metrics_df)}',
        'LLM synthesis would combine the retrieved text with the structured metrics here.',
    ]
    return '\n'.join(parts)

def run_pipeline(user_request: str, ticker: str = CFG.default_ticker, *, k: int = CFG.top_k) -> dict:
    route = route_request(user_request)

    evidence = retrieve_text_evidence(CFG, user_request, k) if route['semantic_search'] else []
    metrics_df = fetch_metrics(conn, ticker) if route['metrics_query'] else pd.DataFrame()
    tables = tabular if (route['spreadsheet_lookup'] or route['visualize']) else {
        'dividends': pd.DataFrame(),
        'stock_prices': pd.DataFrame(),
        'dividend_path': None,
        'stock_path': None,
    }

    response = {
        'request': user_request,
        'ticker': ticker,
        'route': route,
        'text_evidence': evidence,
        'metrics_df': metrics_df,
        'tables': tables,
        'chart_payload': build_chart_payload(metrics_df, tables),
        'answer_stub': build_answer_stub(user_request, route, evidence, metrics_df),
    }
    return response

## **Visualize**

Three charts in one figure:
1. Revenue & Net Income trend (from DuckDB)
2. Dividend-per-share history (from Excel)
3. Adjusted close price, last 5 years (from CSV)

In [ ]:
def plot_dashboard(response: dict) -> None:
    payload = response['chart_payload']
    metrics_df = payload['metrics']
    df_div = payload['dividends']
    df_stock = payload['stock_prices']

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f"{response['ticker']} - Demo Dashboard", fontsize=14, fontweight='bold')

    ax1 = axes[0]
    if not metrics_df.empty:
        for metric, label, color, offset in [
            ('revenue', 'Revenue', '#0078D4', -0.2),
            ('net_income', 'Net Income', '#50C878', 0.2),
        ]:
            subset = metrics_df[metrics_df['metric'] == metric].sort_values('period_year')
            if not subset.empty:
                ax1.bar(subset['period_year'] + offset, subset['value_billions'], width=0.35, label=label, color=color, alpha=0.85)
        ax1.set_title('Revenue & Net Income')
        ax1.set_xlabel('Year')
        ax1.set_ylabel('USD (billions)')
        ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.0f}B'))
        ax1.legend()
    else:
        ax1.text(0.5, 0.5, 'No metrics available', ha='center', va='center', transform=ax1.transAxes, color='gray')
        ax1.set_title('Revenue & Net Income')

    ax2 = axes[1]
    if not df_div.empty:
        date_col = next((c for c in df_div.columns if 'date' in c), None)
        amount_col = next((c for c in df_div.columns if 'amount' in c or 'dividend' in c), None)
        if date_col and amount_col:
            div = df_div[[date_col, amount_col]].copy()
            div[date_col] = pd.to_datetime(div[date_col], errors='coerce')
            div[amount_col] = pd.to_numeric(div[amount_col], errors='coerce')
            div = div.dropna().sort_values(date_col)
            ax2.bar(div[date_col], div[amount_col], width=60, color='#FFB900', alpha=0.85)
            ax2.set_title('Dividend per Share')
            ax2.set_xlabel('Date')
            ax2.set_ylabel('Amount ($)')
            ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.2f}'))
        else:
            ax2.text(0.5, 0.5, 'Dividend columns not recognized', ha='center', va='center', transform=ax2.transAxes, color='gray')
            ax2.set_title('Dividend per Share')
    else:
        ax2.text(0.5, 0.5, 'No dividend data available', ha='center', va='center', transform=ax2.transAxes, color='gray')
        ax2.set_title('Dividend per Share')

    ax3 = axes[2]
    if not df_stock.empty and 'Date' in df_stock.columns and 'Adj Close' in df_stock.columns:
        recent = df_stock[df_stock['Date'] >= '2020-01-01'].copy()
        ax3.plot(recent['Date'], recent['Adj Close'], color='#0078D4', linewidth=1.2)
        ax3.fill_between(recent['Date'], recent['Adj Close'], alpha=0.12, color='#0078D4')
        ax3.set_title('Stock Price (Recent)')
        ax3.set_xlabel('Date')
        ax3.set_ylabel('Adj. Close ($)')
        ax3.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.0f}'))
    else:
        ax3.text(0.5, 0.5, 'No stock data available', ha='center', va='center', transform=ax3.transAxes, color='gray')
        ax3.set_title('Stock Price (Recent)')

    plt.tight_layout()
    plt.show()


## **End-to-End Pipeline**

In [ ]:
demo_request = CFG.default_request
response = run_pipeline(demo_request, ticker=CFG.default_ticker)

print(response['answer_stub'])
print('\nTop evidence preview:')
if response['text_evidence']:
    print(response['text_evidence'][0]['preview'])
else:
    print('No semantic evidence returned.')

if not response['metrics_df'].empty:
    display(response['metrics_df'].pivot(index='period_year', columns='metric', values='value_billions'))
else:
    print('No structured metrics returned.')


In [ ]:
if response['route']['visualize']:
    plot_dashboard(response)
else:
    print('Visualization was not requested for this route.')
